In [2]:
# 04_retrieval_pipeline.ipynb
# Query-Adaptive Retrieval Pipeline Analysis

import json
from pathlib import Path
import pandas as pd

# Use systematic resolution to target the project folder correctly
PROJECT_ROOT = Path("..").resolve()

# ✅ FIX: Pointing to data/processed where the file actually exists
RETRIEVAL_RESULTS_PATH = PROJECT_ROOT / "data" / "processed" / "retrieval_demo_results.json"

# Load retrieval results
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# Run safe verification check
if not RETRIEVAL_RESULTS_PATH.exists():
    print(f"❌ Error: File still not found at {RETRIEVAL_RESULTS_PATH}")
else:
    retrieval_results = load_json(RETRIEVAL_RESULTS_PATH)
    print(f"📦 Pipeline Data Loaded Successfully.")
    print(f"📊 Total Queries Processed: {len(retrieval_results)}")

📦 Pipeline Data Loaded Successfully.
📊 Total Queries Processed: 4


In [4]:
# #Query summary table
query_rows = []

for rec in retrieval_results:
    analysis_block = rec.get("analysis", {})
    # ✅ FIX: Safely dig into the inner nested complexity sub-dictionary
    complexity_block = analysis_block.get("complexity", {})
    features_block = analysis_block.get("features", {})
    
    query_rows.append({
        "query": rec.get("query"),
        "retrieval_mode": rec.get("retrieval_mode"),
        # We drop the missing 'query_type' key or substitute it with structural indicators
        "complexity_score": complexity_block.get("complexity_score", 0.0),
        "complexity_label": complexity_block.get("complexity_label", "medium"),
        "retrieval_hint": complexity_block.get("retrieval_hint", "hybrid"),
        "token_count": features_block.get("token_count", 0),
        "suffix_hits": features_block.get("suffix_match_total", 0),
        "num_results": len(rec.get("results", [])),
    })

query_df = pd.DataFrame(query_rows)
query_df

,query,retrieval_mode,complexity_score,complexity_label,retrieval_hint,token_count,suffix_hits,num_results
0,ሲቢኢ ብር እንዴት እመዝገባለሁ?,hybrid,0.2398,low,sparse_or_hybrid_light,4,0,5
1,የቁጠባ ሂሳብ ለመክፈት ምን ያስፈልጋል?,hybrid,0.2420,low,sparse_or_hybrid_light,5,1,5
2,የATM ካርድ PIN ከረሳሁ ምን ላድርግ?,sparse,0.2774,low,sparse_or_hybrid_light,6,2,5
3,የሞባይል ባንኪንግ ኮሚሽን ስንት ነው?,hybrid,0.2379,low,sparse_or_hybrid_light,5,1,5


In [5]:
#Retrieval mode distribution
query_df["retrieval_mode"].value_counts()

retrieval_mode
hybrid    3
sparse    1
Name: count, dtype: int64

In [7]:
# Inspect one query deeply
idx = 0
sample = retrieval_results[idx]

# Safely extract internal nested blocks
analysis_block = sample.get("analysis", {})
complexity_block = analysis_block.get("complexity", {})
features_block = analysis_block.get("features", {})

print("=========================================================")
print("🔬 DEEP QUERY RETRIEVAL INSPECTION")
print("=========================================================")
print("QUERY:       ", sample.get("query"))
print("MODE:        ", sample.get("retrieval_mode"))
print("COMPLEXITY:  ", complexity_block.get("complexity_label"), f"({complexity_block.get('complexity_score', 0.0):.4f})")
print("HINT:        ", complexity_block.get("retrieval_hint"))
print("---------------------------------------------------------")
print("📊 MORPHOLOGICAL ENGINE MARKERS:")
print("Tokens:      ", features_block.get("token_count", 0))
print("Suffix Hits: ", features_block.get("suffix_match_total", 0))
print("Total Chunks:", len(sample.get("results", [])))
print("=========================================================")

🔬 DEEP QUERY RETRIEVAL INSPECTION
QUERY:        ሲቢኢ ብር እንዴት እመዝገባለሁ?
MODE:         hybrid
COMPLEXITY:   low (0.2398)
HINT:         sparse_or_hybrid_light
---------------------------------------------------------
📊 MORPHOLOGICAL ENGINE MARKERS:
Tokens:       4
Suffix Hits:  0
Total Chunks: 5


In [8]:
#Top reranked chunks for selected query
pd.DataFrame(sample["results"])[[
    "chunk_id", "score", "base_score", "overlap_score", "fintech_bonus", "retrieval_source"
]].head(10)

,chunk_id,score,base_score,overlap_score,fintech_bonus,retrieval_source
0,chunk_000138,0.612500,0.500000,0.25,0.25,hybrid
1,chunk_000098,0.558996,0.459991,0.50,0.25,hybrid
2,chunk_000113,0.542560,0.418900,0.50,0.25,hybrid
3,chunk_000119,0.512500,0.500000,0.25,0.25,hybrid
4,chunk_000040,0.507974,0.332436,0.50,0.25,hybrid


In [9]:
#Show top chunk texts
for i, item in enumerate(sample["results"][:3], start=1):
    print("=" * 80)
    print(f"RANK {i}")
    print("chunk_id:", item["chunk_id"])
    print("score:", item["score"])
    print("source:", item["metadata"]["source_file"])
    print("-" * 80)
    print(item["text"][:1500])
    print()

RANK 1
chunk_id: chunk_000138
score: 0.6125
source: cleaned28_Press_Release_on_IDFR_307ddd5d1c_c5a16feb0d.txt
--------------------------------------------------------------------------------
ዙሮች ለተካሄዱ መርሃ-ግብሮች ከ46.2 ሚሊዮን ብር በላይ ግምት ያላቸው ሽልማቶችን ለደንበኞች በመሸለም ትልቅ የፕሮሞሽን ስራ ተከናውኗል፡፡ ➢ የጥቁር ገበያውን ተፅእኖ ለመቀነስ በሚደረገው ሀገር አቀፍ ጥረት ውስጥ የመፍትሄው አካል ለመሆን እየተሰራ ይገኛል፡፡ በአለም አቀፍ ደረጃ ጁን 16/ሰኔ 9 ቀን የሚከበረውን አለም አቀፍ የሀዋላ ቀንን ባንካችን በተለየ ሁኔታ እንዲከበር ወስኗል፡፡ ቀኑ እንዲዘከር ያስፈለገበት ዋነኛው ምክንያት በሀገር ልማት ውስጥ ወሳኝ ቦታ ያለው የውጪ ሃገር ገንዘብ ግኝታችን በመደበኛው የባንክ ስርአት ውስጥ እንዲያልፍ መልእከት የማስተላለፊያ መድረክ መፍጠር ነው፡፡ ህብረተሰባችንን የማንቃት ተግባር በአግባቡ ከተከናወነ በርካታ ታዳጊ ሃገራት በዘርፉ ከደረሱበት የተጠቃሚነት ደረጃ መድረስ እንደሚቻል የኢትዮጵያ ንግድ ባንክ በፅኑ ያምናል፡፡ 3 ገ ፅ

RANK 2
chunk_id: chunk_000098
score: 0.558996
source: cleaned28_CBE_Noor_Brand_flyer_2_f0213d1a4f_922a1170db (1).txt
--------------------------------------------------------------------------------
ሰ የገንዘብ ማስተላለፍ ሀዋላ አገልግሎት ደንበኞች ከአገር ውስጥ እና ከአገር ውጪ በባንኩ ቅርንጫፎች በኩል የተላከላቸውን ገንዘብ በአቅራቢያቸው ከሚገኝ ማንኛውም የሲቢኢ ኑር ቅርንጫፍ ወይ

In [11]:
# Flatten all retrieval results
flat_rows = []

for rec in retrieval_results:
    analysis_block = rec.get("analysis", {})
    complexity_block = analysis_block.get("complexity", {})
    
    # Safely iterate over the retrieved document results list
    results_list = rec.get("results", [])
    
    for rank, item in enumerate(results_list, start=1):
        # Handle dynamic metadata access patterns securely
        metadata = item.get("metadata", {}) if isinstance(item.get("metadata"), dict) else item
        
        flat_rows.append({
            "query": rec.get("query"),
            "retrieval_mode": rec.get("retrieval_mode"),
            "complexity_label": complexity_block.get("complexity_label", "medium"),
            "rank": rank,
            "chunk_id": item.get("chunk_id") or metadata.get("chunk_id", "N/A"),
            "score": item.get("score"),
            "base_score": item.get("base_score"),
            "overlap_score": item.get("overlap_score"),
            "fintech_bonus": item.get("fintech_bonus"),
            "source_file": metadata.get("source_file") or item.get("source_file", "unknown"),
        })

retrieval_df = pd.DataFrame(flat_rows)
retrieval_df.head()

,query,retrieval_mode,complexity_label,rank,chunk_id,score,base_score,overlap_score,fintech_bonus,source_file
0,ሲቢኢ ብር እንዴት እመዝገባለሁ?,hybrid,low,1,chunk_000138,0.612500,0.500000,0.25,0.25,cleaned28_Press_Release_on_IDFR_307ddd5d1c_c5a...
1,ሲቢኢ ብር እንዴት እመዝገባለሁ?,hybrid,low,2,chunk_000098,0.558996,0.459991,0.50,0.25,cleaned28_CBE_Noor_Brand_flyer_2_f0213d1a4f_92...
2,ሲቢኢ ብር እንዴት እመዝገባለሁ?,hybrid,low,3,chunk_000113,0.542560,0.418900,0.50,0.25,cleaned28_CBE_Noor_Brand_flyer_2_f0213d1a4f_92...
3,ሲቢኢ ብር እንዴት እመዝገባለሁ?,hybrid,low,4,chunk_000119,0.512500,0.500000,0.25,0.25,cleaned28_FAQS amharic cbe.txt
4,ሲቢኢ ብር እንዴት እመዝገባለሁ?,hybrid,low,5,chunk_000040,0.507974,0.332436,0.50,0.25,cleaned28_cbe_0015.txt


In [12]:
#Average score by retrieval mode
retrieval_df.groupby("retrieval_mode")["score"].mean().reset_index()

,retrieval_mode,score
0,hybrid,0.534695
1,sparse,2.193621


In [13]:
#Paper-ready retrieval summary
paper_retrieval_summary = pd.DataFrame([{
    "Queries tested": len(retrieval_results),
    "Sparse selected": int((query_df["retrieval_mode"] == "sparse").sum()),
    "Dense selected": int((query_df["retrieval_mode"] == "dense").sum()),
    "Hybrid selected": int((query_df["retrieval_mode"] == "hybrid").sum()),
    "Avg retrieved chunks/query": round(query_df["num_results"].mean(), 2),
}])

paper_retrieval_summary

,Queries tested,Sparse selected,Dense selected,Hybrid selected,Avg retrieved chunks/query
0,4,1,0,3,5.0
